Fundamentals of Deep Learning Models

# Lab 09-1: Transformer Model for Language Understanding
## Exercise: Translator for Portuguese to English

This exercise implements a **Transformer-based machine translation model** (Sections 9.1–9.7) that translates Portuguese sentences into English. The implementation covers the full Transformer architecture, including positional encoding (Section 9.4, Eq. 9.18), scaled dot-product attention (Section 9.2, Eq. 9.19), multi-head attention (Section 9.6), encoder and decoder attention blocks with residual connections and layer normalization (Section 9.5), and attention masking techniques including padding masks and causal masks (Section 9.7, Eq. 9.21).

This exercise is based on the TensorFlow Tutorials and has been significantly modified for educational purposes.
Reference: https://www.tensorflow.org/text/tutorials/transformer

### Load Libraries

In [1]:
RunningInCOLAB = 'google.colab' in str(get_ipython())

if RunningInCOLAB:
    from tqdm.notebook import tqdm
else:
    from tqdm import tqdm

import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

print('NumPy version:', np.__version__)
print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)
print('TensorFlow Datasets version:', tfds.__version__)
print('Matplotlib version:', plt.matplotlib.__version__)

# Import tf_text to load the ops used by the tokenizer saved model
# For tensorflow=2.10, tensorflow-text=2.10 & protobuf=3.20.0 is necessary.
import tensorflow_text  # pylint: disable=unused-import

# Check available GPU devices
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))

NumPy version: 1.26.4
TensorFlow version: 2.16.1
Keras version: 3.4.1
TensorFlow Datasets version: 4.9.9
Matplotlib version: 3.9.1
Num GPUs Available:  1


### Load the Dataset

Use [TensorFlow datasets](https://www.tensorflow.org/datasets) to load the [Portuguese-English translation dataset](https://github.com/neulab/word-embeddings-for-nmt) from the [TED Talks Open Translation Project](https://www.ted.com/participate/translate).

This dataset contains approximately 50000 training examples, 1100 validation examples, and 2000 test examples.

In [2]:
(ds_test, ds_train, ds_val), ds_info = tfds.load('ted_hrlr_translate/pt_to_en',
                                                 split=['test', 'train', 'validation'],
                                                 as_supervised=True,
                                                 with_info=True)

print(ds_info.features)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

NonMatchingChecksumError: Artifact http://www.phontron.com/data/qi18naacl-dataset.tar.gz, downloaded to /home/swx/tensorflow_datasets/downloads/ted_hrlr_translate/phontron.com_qi18naacl-datasetLVhDyvf9PQ-GjP4jim31ESZuvRjHI5wpNhR5SJvsNOo.tar.gz.tmp.d424b9bf44d74fdea96c28e45d5c06b0/index.html, has wrong checksum:
* Expected: UrlInfo(size=124.94 MiB, checksum='216a86c3df4d4f522856fe9b920ff5be6b394d769cc88974ae8f9f5546953bbc', filename='qi18naacl-dataset.tar.gz')
* Got: UrlInfo(size=705 bytes, checksum='9e1cd18a546ca4ad9b8452b8f251289da1f27f58978964487bc84f791641e58f', filename='index.html')
To debug, see: https://www.tensorflow.org/datasets/overview#fixing_nonmatchingchecksumerror

The `tf.data.Dataset` object returned by TensorFlow datasets yields pairs of text examples:

In [ ]:
for pt_examples, en_examples in ds_train.batch(3).take(1):
    for pt in pt_examples.numpy():
        print(pt.decode('utf-8'))

    print()

    for en in en_examples.numpy():
        print(en.decode('utf-8'))

### Text tokenization & detokenization

One popular implementation is demonstrated in the [Subword tokenizer tutorial](https://www.tensorflow.org/text/guide/subwords_tokenizer) builds subword tokenizers (`text.BertTokenizer`) optimized for this dataset and exports them in a [saved_model](https://www.tensorflow.org/guide/saved_model).

Download and unzip and import the `saved_model`: <br>
The `tf.saved_model` contains two text tokenizers, one for English and one for Portuguese. Both have the same methods:

In [ ]:
model_name = 'ted_hrlr_translate_pt_en_converter'
keras.utils.get_file(
    f'{model_name}.zip',
    f'https://storage.googleapis.com/download.tensorflow.org/models/{model_name}.zip',
    cache_dir='.', cache_subdir='', extract=True
)


In [ ]:
tokenizers = tf.saved_model.load(model_name)

[item for item in dir(tokenizers.en) if not item.startswith('_')]

The `tokenize` method converts a batch of strings to a padded-batch of token IDs. This method splits punctuation, lowercases and unicode-normalizes the input before tokenizing. That standardization is not visible here because the input data is already standardized.

In [ ]:
for en in en_examples.numpy():
    print(en.decode('utf-8'))

encoded = tokenizers.en.tokenize(en_examples)

print()

for row in encoded.to_list():
    print(row)

The `detokenize` method attempts to convert these token IDs back to human readable text:

In [ ]:
round_trip = tokenizers.en.detokenize(encoded)
for line in round_trip.numpy():
    print(line.decode('utf-8'))

The lower level `lookup` method converts from token-IDs to token text:

In [ ]:
tokens = tokenizers.en.lookup(encoded)
tokens

Here you can see the "subword" aspect of the tokenizers. The word "searchability" is decomposed into "search ##ability" and the word "serendipity" into "s ##ere ##nd ##ip ##ity"

Now take a minute to investigate the distribution of tokens per example in the dataset:

In [ ]:
lengths = []

for pt_examples, en_examples in ds_train.batch(1024):
    pt_tokens = tokenizers.pt.tokenize(pt_examples)
    lengths.append(pt_tokens.row_lengths())

    en_tokens = tokenizers.en.tokenize(en_examples)
    lengths.append(en_tokens.row_lengths())
    print('.', end='', flush=True)

In [ ]:
all_lengths = np.concatenate(lengths)

plt.figure(figsize=(4,2))
plt.hist(all_lengths, np.linspace(0, 500, 101))
plt.ylim(plt.ylim())
max_length = max(all_lengths)
plt.plot([max_length, max_length], plt.ylim())
plt.title(f'Max tokens per example: {max_length}');

### Setup input pipeline

To build an input pipeline suitable for training define some functions to transform the dataset.

In [ ]:
MAX_TOKENS = 128

def tokenize_pairs(pt, en):  # tokenize (pt, en) pair
    pt = tokenizers.pt.tokenize(pt)
    pt = pt[:, :MAX_TOKENS] # Trim to MAX_TOKENS.
    pt = pt.to_tensor()     # Convert from ragged to dense, padding with zeros.

    en = tokenizers.en.tokenize(en)
    en = en[:, :(MAX_TOKENS+1)]   # Trim to MAX_TOKENS.

    # Convert from ragged to dense, padding with zeros.
    en_inputs = en[:, :-1].to_tensor()  # Drop the [END] tokens
    en_labels = en[:, 1:].to_tensor()   # Drop the [START] tokens

    return (pt, en_inputs), en_labels

Here's a simple input pipeline that processes, shuffles and batches the data:

In [ ]:
BUFFER_SIZE = ds_info.splits['train'].num_examples
BATCH_SIZE = 64

def make_batches(ds):
    return (ds
           .cache()
           .shuffle(BUFFER_SIZE)
           .batch(BATCH_SIZE, drop_remainder=True)
           .map(tokenize_pairs, num_parallel_calls=tf.data.AUTOTUNE)
           .prefetch(tf.data.AUTOTUNE))

train_batches = make_batches(ds_train)
val_batches = make_batches(ds_val)

## Implementation of Transformer
### Block Diagram of Transformer

The following diagram illustrates the overall Transformer architecture (Section 9.3, Figure 9.4).

<img src="https://www.tensorflow.org/images/tutorials/transformer/transformer.png" width="400" alt="transformer">

### Positional encoding

The positional encoding formula uses sinusoidal functions at geometrically spaced frequencies to inject position information into the input embeddings (Section 9.4, Eq. 9.18):

$$
\text{PE}(t, 2j) = \sin\!\left(\frac{t}{B^{2j/D}}\right), \quad \text{PE}(t, 2j+1) = \cos\!\left(\frac{t}{B^{2j/D}}\right)
$$

where $t$ is the position index, $j$ is the dimension pair index, $D = d_{\text{model}}$ is the embedding dimension, and $B = 10000$ is the base constant. The final result is a matrix of shape $(1, T, D)$ where `sin` and `cos` values are interleaved along the last dimension.

In [ ]:
def positional_encoding(length, d_model):
    """
    Precomputes a matrix with all the positional encodings

    Arguments:
        length (int) -- Maximum number of positions to be encoded
        d_model (int) -- Encoding size

    Returns:
        pos_encoding -- (1, length, d_model) A matrix with the positional encodings
    """

    positions = np.arange(length)[:, np.newaxis]  # position index
    depths = np.arange(d_model//2)[np.newaxis, :] # depth index = depth // 2

    ### START CODE HERE ###

    # initialize a matrix angle_rads of all the angles
    depths = None                                 # depths = 10000^(2*(i//2)/d_model)
    angle_rads = None                             # pos/depths (length,d_model/2)

    pos_sin = None                                # apply sin function (length,d_model/2,1)
    pos_cos = None                                # apply cos function (length,d_model/2,1)

    # concatenate sin to even indices in the array; 2i
    # and cos to odd indices in the array; 2i+1
    pos_encoding = None                           # interleave pos_sin and pos_cos (length,d_model)

    ### END CODE HERE ###

    pos_encoding = pos_encoding.reshape(1, length, d_model)

    return tf.cast(pos_encoding, dtype=tf.float32)

In [ ]:
n, d = 2048, 512
pos_encoding = positional_encoding(n, d)
print(pos_encoding.shape)
pos_encoding = pos_encoding[0]

# Juggle the dimensions for the plot
pe_rearrange = tf.reshape(pos_encoding, (n, d//2, 2))
pe_rearrange = tf.transpose(pe_rearrange, (2, 1, 0))
pe_rearrange = tf.reshape(pe_rearrange, (d, n))

plt.figure(figsize=(6,4))
plt.pcolormesh(pe_rearrange, cmap='RdBu')
plt.ylabel('Depth')
plt.xlabel('Position')
plt.colorbar()
plt.show()

print('This graph is only to show that each position has unique signiture.')

By definition these vectors align well with nearby vectors along the position axis. Below the position encoding vectors are normalized and the vector from position `1000` is compared, by dot-product, to all the others:

In [ ]:
pos_encoding/=tf.norm(pos_encoding, axis=1, keepdims=True)
p = pos_encoding[1000]
dots = tf.einsum('pd,d -> p', pos_encoding, p)
plt.figure(figsize=(6,4))
plt.subplot(2,1,1)
plt.plot(dots)
plt.ylim([0,1])
plt.plot([950, 950, float('nan'), 1050, 1050],
         [0,1,float('nan'),0,1], color='k', label='Zoom')
plt.legend()
plt.subplot(2,1,2)
plt.plot(dots)
plt.xlim([950, 1050])
plt.ylim([0,1])


### Scaled Dot-Product Attention

The Transformer uses **scaled dot-product attention** (Section 9.2, Eq. 9.14) to compute relevance-weighted representations. With an optional additive mask $\mathbf{M}$ for padding or causal masking (Section 9.7, Eq. 9.21), the masked attention is:

$$
\text{MaskedAttention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}_{\text{row}}\!\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V} \quad \text{(Eq. 9.21)}
$$

where $\mathbf{Q}$, $\mathbf{K}$, $\mathbf{V}$ are the query, key, and value matrices, $d_k$ is the key dimension, and $\mathbf{M}$ contains $0$ for allowed positions and large negative values (e.g., $-10^9$) for masked positions.

Implement the function `scaled_dot_product_attention()` to create attention-based representations.

**Hints:**
- Use [tf.matmul](https://www.tensorflow.org/api_docs/python/tf/linalg/matmul) for matrix multiplication.
- The mask uses `1` for allowed and `0` for blocked positions; convert blocked positions to $-10^9$ before adding to scores.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Compute scaled dot-product attention (Eq. 9.19/9.21).

    With multi-head attention, input shapes are (batch, head, seq_len, depth).
    Mask shapes are (batch, head, seq_len_q, seq_len_k).

    Arguments:
        q -- query shape == (..., seq_len_q, depth)
        k -- key shape == (..., seq_len_k, depth)
        v -- value shape == (..., seq_len_v, depth_v)
        mask -- float tensor broadcastable to (..., seq_len_q, seq_len_k). Defaults to None.

    Returns:
        output -- weighted sum of values, shape (..., seq_len_q, depth_v)
        attention_weights -- softmax weights, shape (..., seq_len_q, seq_len_k)
    """
    ### START CODE HERE ###

    # Compute Q*K^T: dot product of queries and keys (Eq. 9.19)
    matmul_qk = None            # shape (..., seq_len_q, seq_len_k)

    # Scale by sqrt(d_k) to prevent softmax saturation (Eq. 9.14)
    dk = None                   # depth of the key vectors
    scaled_attention_logits = None

    # Apply additive mask: convert (1=keep, 0=block) to (0, -1e9) (Eq. 9.21)
    if mask is not None:
        mask = tf.cast(mask, dtype=tf.float32)
        scaled_attention_logits += None

    # Apply softmax along key dimension to get attention weights
    attention_weights = None

    # Weighted sum of value vectors (Eq. 9.12)
    output = None

    ### END CODE HERE ###

    return output, attention_weights

In [ ]:
temp_k = tf.constant([[10, 0, 0],
                      [0, 10, 0],
                      [0, 0, 10],
                      [0, 0, 10]], dtype=tf.float32)  # (4, 3)

temp_v = tf.constant([[1, 0],
                      [10, 0],
                      [100, 5],
                      [1000, 6]], dtype=tf.float32)  # (4, 2)

# This `query` aligns with the second `key`,
# so the second `value` is returned.
temp_q = tf.constant([[0, 10, 0]], dtype=tf.float32)  # (1, 3)

temp_out, temp_attn = scaled_dot_product_attention(temp_q, temp_k, temp_v, None)
print('Attention weights are:')
print(tf.where(temp_attn<1e-9,0.,temp_attn))
print('Output is:')
print(tf.where(temp_out<1e-9,0.,temp_out))

**Expected Output**
```
Attention weights are:
tf.Tensor([[0. 1. 0. 0.]], shape=(1, 4), dtype=float32)
Output is:
tf.Tensor([[10.  0.]], shape=(1, 2), dtype=float32)
```

In [ ]:
# This query aligns with a repeated key (third and fourth),
# so all associated values get averaged.
temp_q = tf.constant([[0, 0, 10]], dtype=tf.float32)  # (1, 3)

temp_out, temp_attn = scaled_dot_product_attention(temp_q, temp_k, temp_v, None)
print('Attention weights are:')
print(tf.where(temp_attn<1e-9,0.,temp_attn))
print('Output is:')
print(tf.where(temp_out<1e-9,0.,temp_out))

**Expected Output**
```
Attention weights are:
tf.Tensor([[0.  0.  0.5 0.5]], shape=(1, 4), dtype=float32)
Output is:
tf.Tensor([[550.    5.5]], shape=(1, 2), dtype=float32)
```

In [ ]:
# This query aligns equally with the first and second key,
# so their values get averaged.
temp_q = tf.constant([[10, 10, 0]], dtype=tf.float32)  # (1, 3)

temp_out, temp_attn = scaled_dot_product_attention(temp_q, temp_k, temp_v, None)
print('Attention weights are:')
print(tf.where(temp_attn<1e-9,0.,temp_attn))
print('Output is:')
print(tf.where(temp_out<1e-9,0.,temp_out))

**Expected Output**
```
Attention weights are:
tf.Tensor([[0.5 0.5 0.  0. ]], shape=(1, 4), dtype=float32)
Output is:
tf.Tensor([[5.5 0. ]], shape=(1, 2), dtype=float32)
```

In [ ]:
temp_q = tf.constant([[0, 0, 10],
                      [0, 10, 0],
                      [10, 10, 0]], dtype=tf.float32)  # (3, 3)

temp_out, temp_attn = scaled_dot_product_attention(temp_q, temp_k, temp_v, None)
print('Attention weights are:')
print(tf.where(temp_attn<1e-9,0.,temp_attn))
print('Output is:')
print(tf.where(temp_out<1e-9,0.,temp_out))

**Expected Output**
```
Attention weights are:
tf.Tensor(
[[0.  0.  0.5 0.5]
 [0.  1.  0.  0. ]
 [0.5 0.5 0.  0. ]], shape=(3, 4), dtype=float32)
Output is:
tf.Tensor(
[[550.    5.5]
 [ 10.    0. ]
 [  5.5   0. ]], shape=(3, 2), dtype=float32)
```

In [ ]:
q = np.array([[1, 0, 1, 1], [0, 1, 1, 1], [1, 0, 0, 1]]).astype(np.float32)
k = np.array([[1, 1, 0, 1], [1, 0, 1, 1 ], [0, 1, 1, 0], [0, 0, 0, 1]]).astype(np.float32)
v = np.array([[0, 0], [1, 0], [1, 0], [1, 1]]).astype(np.float32)

attention, weights = scaled_dot_product_attention(q, k, v, None)

print('Weights:', weights)
print('weights shape is [q.shape[0], k.shape[0]] =', [q.shape[0], k.shape[0]])
print('Attention:', attention)
print('attention shape is [q.shape[0], v.shape[1]] =', (q.shape[0], v.shape[1]))

mask = np.array([[1, 1, 0, 1], [1, 1, 0, 1], [1, 1, 0, 1]])

attention, weights = scaled_dot_product_attention(q, k, v, mask)
print('Masked MHA results:')
print('Weights:', weights)
print('Attention:', attention)

**Expected Output**
```
Weights: tf.Tensor(
[[0.25894776 0.42693266 0.15705977 0.15705977]
 [0.2772748  0.2772748  0.2772748  0.16817565]
 [0.33620113 0.33620113 0.12368149 0.2039163 ]], shape=(3, 4), dtype=float32)
weights shape is [q.shape[0], k.shape[0]] = [3, 4]
Attention: tf.Tensor(
[[0.7410522  0.15705977]
 [0.7227252  0.16817565]
 [0.6637989  0.2039163 ]], shape=(3, 2), dtype=float32)
attention shape is [q.shape[0], v.shape[1]] = (3, 2)
Masked MHA results:
Weights: tf.Tensor(
[[0.30719587 0.5064804  0.         0.18632373]
 [0.38365173 0.38365173 0.         0.23269653]
 [0.38365173 0.38365173 0.         0.23269653]], shape=(3, 4), dtype=float32)
Attention: tf.Tensor(
[[0.6928041  0.18632373]
 [0.61634827 0.23269653]
 [0.61634827 0.23269653]], shape=(3, 2), dtype=float32)
```

### Multi-Head Attention

Multi-head attention (Section 9.6) projects queries, keys, and values into $h$ parallel heads, each operating on a reduced dimension $d_{\text{head}} = d_{\text{model}} / h$, then concatenates the results and applies a final linear projection (Figure 9.8).

A `MultiHeadAttention` layer to try out. At each location in the sequence, `y`, the `MultiHeadAttention` runs all 8 attention heads across all other locations in the sequence, returning a new vector of the same length at each location.

In [ ]:
USE_TF_MHA = True

class MultiHeadAttention(keras.layers.Layer):
    def __init__(self,*, num_heads, key_dim):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads          # number of heads
        self.depth = key_dim                # key dimension for a single head
        self.d_model = num_heads * key_dim  # model dimension; not input (QKV) dimension

        self.wq = keras.layers.Dense(self.d_model)
        self.wv = keras.layers.Dense(self.d_model)
        self.wk = keras.layers.Dense(self.d_model)
        self.wo = keras.layers.Dense(self.d_model)

    def split_heads(self, x, batch_size):
        """Split the last dimension into (num_heads, depth).
        Transpose the result such that the shape is (batch_size, num_heads, seq_len, depth)
        """
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        x = tf.transpose(x, perm=[0, 2, 1, 3])
        return x

    def call(self, query, value, key, attention_mask=None, return_attention_scores=False):
        batch_size = tf.shape(query)[0]
        seq_length = tf.shape(query)[1]

        q = self.wq(query)          # (b, seq_len, d_input_q) -> (b, seq_len, d_model)
        k = self.wk(key)            # (b, seq_len, d_input_k) -> (b, seq_len, d_model)
        v = self.wv(value)          # (b, seq_len, d_input_v) -> (b, seq_len, d_model)

        ### START CODE HERE ###

        # depth = d_model / num_heads
        q = None                    # (batch_size, num_heads, seq_len_q, depth)
        k = None                    # (batch_size, num_heads, seq_len_k, depth)
        v = None                    # (batch_size, num_heads, seq_len_v, depth)

        # scaled_attention.shape == (batch_size, num_heads, seq_len_q, depth)
        # attention_weights.shape == (batch_size, num_heads, seq_len_q, seq_len_k)
        scaled_attention, attention_weights = None

        # (batch_size, seq_len_q, num_heads, depth)
        scaled_attention = None

        # (batch_size, seq_len_q, d_model)
        concat_attention = None

        ### END CODE HERE ###

        output = self.wo(concat_attention)  # (batch_size, seq_len_q, d_model)

        if return_attention_scores:
            return output, attention_weights
        else:
            return output

In [ ]:
tf.random.set_seed(1)

depth = 512 // 8

temp_mha = MultiHeadAttention(num_heads=8, key_dim=depth)

y = tf.random.uniform((1, 60, 512))  # (batch_size, encoder_sequence, d_model)

out, attn = temp_mha(query=y, value=y, key=y, attention_mask=None, return_attention_scores=True)

print(out.shape, attn.shape)
print('output:\n', out[0,1,2:5])
print('attention:\n', attn[0,1,2,3:5])

**Expected Output**
```
(1, 60, 512) (1, 8, 60, 60)
output:
 tf.Tensor([-0.44711024 -0.08185582  0.8300966 ], shape=(3,), dtype=float32)
attention:
 tf.Tensor([0.02097543 0.01911536], shape=(2,), dtype=float32)
```

#### Checking dimensions in MultiHeadAttention

In [ ]:
mha = keras.layers.MultiHeadAttention(num_heads=2, key_dim=9, value_dim=8, output_shape=(7,))

x_q = tf.random.normal((1,11,12))
x_v = tf.random.normal((1,13,14))
x_k = tf.random.normal((1,13,15))

print('Input Q (b,qs,qf):', x_q.shape)
print('Input V (b,vs=ks,vf):', x_v.shape)
print('Input K (b,ks=vs,kf):', x_k.shape)

y_a, y_s = mha(query=x_q, value=x_v, key=x_k, return_attention_scores=True)

print('Output shape (b,qs,od):', y_a.shape)
print('Attn Score shape (b,h,qs,ks=vs):', y_s.shape)

w = mha.get_weights()

print('Query prj shape (qf,h,kd)', w[0].shape)
print('Key prj shape (kf,h,kd)', w[2].shape)
print('Value prj shape (vf,h,vd)', w[4].shape)
print('Output prj shape (h,vd,od)', w[6].shape)


### Point wise feed forward network

The position-wise feedforward network consists of two fully-connected layers with a ReLU activation in between (Section 9.5, Eq. 9.20):

$$\text{Feedforward}(\mathbf{x}) = \mathbf{W}_2 \,\text{ReLU}(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$$

In the original Transformer, $d_{\text{FF}} = 2048$ and $d_{\text{model}} = 512$ (i.e., $d_{\text{FF}} = 4 \times d_{\text{model}}$).

In [ ]:
def feed_forward_network(d_model, dff):
    return keras.Sequential([
        keras.layers.Dense(dff, activation='relu'),  # (batch_size, seq_len, dff)
        keras.layers.Dense(d_model)  # (batch_size, seq_len, d_model)
    ])

In [ ]:
tf.random.set_seed(1)

sample_ffn = feed_forward_network(512, 2048)
sample_ffn_output = sample_ffn(tf.random.uniform((64, 50, 512)))

print(sample_ffn_output.shape)
print(sample_ffn_output[1,2,3:5])

**Expected Output**
```
(64, 50, 512)
tf.Tensor([-0.08358504  0.03292187], shape=(2,), dtype=float32)
```

### Encoder and decoder

A transformer model follows the same general pattern as a standard [sequence to sequence with attention model](https://www.tensorflow.org/text/tutorials/nmt_with_attention.ipynb).

* The input sentence is passed through `N` encoder layers that generates an output for each token in the sequence.
* The decoder attends to the encoder's output and its own input (self-attention) to predict the next word.

### Encoder layer

Each encoder layer (Section 9.5, Figure 9.7a) consists of two sublayers:

1. Multi-head self-attention (with padding mask)
2. Position-wise feedforward network (Eq. 9.20)

Each sublayer has a residual connection followed by layer normalization. The output of each sublayer is $\text{LayerNorm}(\mathbf{x} + \text{Sublayer}(\mathbf{x}))$. Dropout is applied after each sublayer before the residual addition.

In [ ]:
class EncoderLayer(keras.layers.Layer):
    def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
        super(EncoderLayer, self).__init__()

        depth = d_model // num_heads

        if USE_TF_MHA:
            self.mha = keras.layers.MultiHeadAttention(num_heads=num_heads,
                                                          key_dim=depth,
                                                          attention_axes=1)
        else:
            self.mha = MultiHeadAttention(num_heads=num_heads, key_dim=depth)
        self.ffn = feed_forward_network(d_model, dff)

        self.lyrnorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.lyrnorm2 = keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = keras.layers.Dropout(dropout_rate)
        self.dropout2 = keras.layers.Dropout(dropout_rate)

    def call(self, x, mask=None):

        ### START CODE HERE ###

        # Self-attention: Q=K=V=x (Section 9.5)
        attn_out = None                     # (batch_size, input_seq_len, d_model)
        # Dropout + residual + LayerNorm
        drpout1 = None
        out1 = None

        # Feedforward sub-block (Eq. 9.20)
        ffn_out = None
        drpout2 = None
        out2 = None                         # LayerNorm(x + Sublayer(x))

        ### END CODE HERE ###

        return out2

In [ ]:
tf.random.set_seed(1)

sample_encoder_layer = EncoderLayer(d_model=512, num_heads=8, dff=2048)

sample_encoder_layer_output = sample_encoder_layer(
    tf.random.uniform((64, 43, 512)), mask=None)

print(sample_encoder_layer_output.shape)  # (batch_size, input_seq_len, d_model)
print(sample_encoder_layer_output[1,2,3:5])

**Expected Output**
```
(64, 43, 512)
tf.Tensor([-0.6090227   0.01429609], shape=(2,), dtype=float32)
```

### Encoder

The `Encoder` (Section 9.3) consists of:
1. Input embedding (scaled by $\sqrt{d_{\text{model}}}$)
2. Positional encoding (Eq. 9.18)
3. $N$ encoder layers (Figure 9.7a)
4. Dropout after the embedding + positional encoding sum

The input is put through an embedding which is summed with the positional encoding. The output of this summation is the input to the encoder layers.

In [ ]:
class Encoder(keras.layers.Layer):
    def __init__(self,*, num_layers, d_model, num_heads, dff, input_vocab_size,
                 dropout_rate=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding_scale = tf.math.sqrt(tf.cast(self.d_model, tf.float32))

        self.embedding = keras.layers.Embedding(input_vocab_size, d_model, mask_zero=False)
        self.pos_encoding = positional_encoding(MAX_TOKENS, self.d_model)

        self.enc_layers = [
            EncoderLayer(d_model=d_model, num_heads=num_heads, dff=dff, dropout_rate=dropout_rate)
            for _ in range(num_layers)]

        self.dropout = keras.layers.Dropout(dropout_rate)

    def call(self, x, mask=None):

      seq_len = tf.shape(x)[1]

      ### START CODE HERE ###

      # Embedding + positional encoding (Section 9.3)
      x = None                    # (batch_size, input_seq_len, d_model)
      x *= None                   # Scale by sqrt(d_model)
      x += None                   # Add positional encoding (Eq. 9.18)

      x = None                    # dropout after embedding+positional encoding

      ### END CODE HERE ###

      for i in range(self.num_layers):
          x = self.enc_layers[i](x, mask=mask)

      return x  # (batch_size, input_seq_len, d_model)

In [ ]:
tf.random.set_seed(1)

sample_encoder = Encoder(num_layers=2, d_model=512, num_heads=8,
                         dff=2048, input_vocab_size=8500)
temp_input = tf.random.uniform((64, 62), dtype=tf.int64, minval=0, maxval=200)

sample_encoder_output = sample_encoder(temp_input, mask=None)

print(sample_encoder_output.shape)  # (batch_size, input_seq_len, d_model)
print(sample_encoder_output[1,2,3:5])

**Expected Output**
```
(64, 62, 512)
tf.Tensor([-2.2488644  1.6115965], shape=(2,), dtype=float32)
```

### Decoder layer

Each decoder layer (Section 9.5, Figure 9.7b) consists of three sublayers:

1. Masked multi-head self-attention (with causal mask and padding mask)
2. Multi-head cross-attention (with padding mask). $\mathbf{V}$ and $\mathbf{K}$ receive the encoder output; $\mathbf{Q}$ receives the output from the masked self-attention sublayer.
3. Position-wise feedforward network (Eq. 9.20)

Each sublayer has a residual connection followed by layer normalization and dropout. The cross-attention weights represent the importance given to each encoder token for predicting the current decoder output.

In [ ]:
class DecoderLayer(keras.layers.Layer):
    def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
        super(DecoderLayer, self).__init__()

        depth = d_model // num_heads

        if USE_TF_MHA:
            self.mha1 = keras.layers.MultiHeadAttention(num_heads=num_heads,
                                                           key_dim=depth,
                                                           attention_axes=1)
            self.mha2 = keras.layers.MultiHeadAttention(num_heads=num_heads,
                                                           key_dim=depth,
                                                           attention_axes=1)
        else:
            self.mha1 = MultiHeadAttention(num_heads=num_heads, key_dim=depth)
            self.mha2 = MultiHeadAttention(num_heads=num_heads, key_dim=depth)

        self.ffn = feed_forward_network(d_model, dff)

        self.lyrnorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.lyrnorm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.lyrnorm3 = keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = keras.layers.Dropout(dropout_rate)
        self.dropout2 = keras.layers.Dropout(dropout_rate)
        self.dropout3 = keras.layers.Dropout(dropout_rate)

        self.last_attn_scores = None

    def call(self, x, enc_output, look_ahead_mask=None, padding_mask=None):

        ### START CODE HERE ###

        # Masked self-attention with causal mask (Section 9.7)
        attn1 = None
        drpout1 = None
        out1 = None

        # enc_output.shape == (batch_size, input_seq_len, d_model)
        attn2, attn_scores = self.mha2(...)   # q,v,k,m (batch_size, target_seq_len, d_model)
        drpout2 = None
        out2 = None                           # (batch_size, target_seq_len, d_model)

        ffn_out = None                        # (batch_size, target_seq_len, d_model)
        drpout3 = None
        out3 = None                           # (batch_size, target_seq_len, d_model)

        ### END CODE HERE ###

        self.last_attn_scores = attn_scores

        return out3             # return weights for demonstration purpose later

In [ ]:
tf.random.set_seed(1)

sample_decoder_layer = DecoderLayer(d_model=512, num_heads=8, dff=2048)

sample_decoder_layer_output = sample_decoder_layer(
    tf.random.uniform((64, 50, 512)), sample_encoder_layer_output,
    look_ahead_mask=None, padding_mask=None)

print(sample_decoder_layer_output.shape)  # (batch_size, target_seq_len, d_model)
print(sample_decoder_layer_output[1,2,3:5])

**Expected Output**
```
(64, 50, 512)
tf.Tensor([-0.86408186  1.449299  ], shape=(2,), dtype=float32)
```

### Decoder

The `Decoder` (Section 9.3) consists of:
1. Output embedding (scaled by $\sqrt{d_{\text{model}}}$)
2. Positional encoding (Eq. 9.18)
3. $N$ decoder layers (Figure 9.7b)

The target is put through an embedding which is summed with the positional encoding. The output of this summation is the input to the decoder layers. The output of the decoder is the input to the final linear layer.

In [ ]:
class Decoder(keras.layers.Layer):
    def __init__(self,*, num_layers, d_model, num_heads, dff, target_vocab_size,
                 dropout_rate=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = keras.layers.Embedding(target_vocab_size, d_model, mask_zero=False)
        self.pos_encoding = positional_encoding(MAX_TOKENS, d_model)

        self.dec_layers = [
            DecoderLayer(d_model=d_model, num_heads=num_heads, dff=dff, dropout_rate=dropout_rate)
            for _ in range(num_layers)]
        self.dropout = keras.layers.Dropout(dropout_rate)

        self.last_attn_scores = None

    def call(self, x, enc_output, look_ahead_mask=None, padding_mask=None):

        seq_len = tf.shape(x)[1]

        ### START CODE HERE ###

        # Embedding + scaling + positional encoding (Section 9.3)
        x = None                          # (batch_size, target_seq_len, d_model)
        x *= None                         # Scale by sqrt(d_model)
        x += None                         # Add positional encoding (Eq. 9.18)

        x = None                          # dropout after embedding+positional encoding

        ### END CODE HERE ###

        for i in range(self.num_layers):
            x = self.dec_layers[i](x, enc_output, look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)

        self.last_attn_scores = self.dec_layers[-1].last_attn_scores

        # x.shape == (batch_size, target_seq_len, d_model)
        return x

In [ ]:
tf.random.set_seed(1)

sample_decoder = Decoder(num_layers=2, d_model=512, num_heads=8,
                         dff=2048, target_vocab_size=8000)
temp_input = tf.random.uniform((64, 26), dtype=tf.int64, minval=0, maxval=200)

output = sample_decoder(temp_input,
                        enc_output=sample_encoder_output,
                        look_ahead_mask=None,
                        padding_mask=None)

print(output.shape, sample_decoder.last_attn_scores.shape)
print('output:\n', output[1,2,3:5])
print('attention:\n', sample_decoder.last_attn_scores[1,:,3,4])

**Expected Output**
```
(64, 26, 512) (64, 8, 26, 62)
output:
 tf.Tensor([-1.1841264 -1.2807183], shape=(2,), dtype=float32)
attention:
 tf.Tensor(
[0.01539868 0.01678353 0.01620763 0.01663938 0.01581496 0.01589628
 0.01628351 0.01579851], shape=(8,), dtype=float32)
```

### Create the transformer model

A transformer consists of the encoder, decoder, and a final linear layer. The output of the decoder is the input to the linear layer and its output is returned.

Mask all the pad tokens in the batch of sequence. It ensures that the model does not treat padding as the input. Since the sequence has zero pads after end of sentence, create a mask to exclude non-sentence area.<br>
The mask have 1s for elements to use.

In [ ]:
def create_padding_mask(seq):
    '''
    The output has a form of (batch, head, seq_len_q, seq_len_k).
    The padding mask is applied to sequence k direction.
    '''
    ### START CODE HERE ###

    # Create binary mask: 1 for real tokens, 0 for padding (Section 9.7)
    padding_mask = None             # (b, s)
    # Expand dims for broadcasting to (batch, 1, 1, seq_len_k)
    padding_mask = None             # (batch_size, 1, 1, seq_len_k)
    padding_mask = None             # Typecast to float: 1. for real tokens, 0. for padding (Section 9.7)

    ### END CODE HERE ###

    return padding_mask

In [ ]:
x = tf.constant([[7, 6, 0, 0, 1], [1, 2, 3, 0, 0], [0, 0, 0, 4, 5]])
temp = create_padding_mask(x)
print(temp)

**Expected Output**
```
tf.Tensor(
[[[[1. 1. 0. 0. 1.]]]
 [[[1. 1. 1. 0. 0.]]]
 [[[0. 0. 0. 1. 1.]]]], shape=(3, 1, 1, 5), dtype=float32)
```

The look-ahead mask (causal mask) is used to prevent the decoder from attending to future tokens during autoregressive generation (Section 9.7, Figure 9.9c). Combined with the padding mask, it ensures that each position can only attend to itself and preceding non-padding tokens (Figure 9.11).
The mask uses `1` for positions to attend to and `0` for positions to block.

In [ ]:
def create_look_ahead_mask(seq):
    '''
    The output has a form of (batch, head, seq_len_q, seq_len_k).
    And here seq_len_q = seq_len_k = t (target length).
    '''
    
    ### START CODE HERE ###

    # Padding mask: 1 for real tokens, 0 for padding
    padding_mask = None                # (b, t)
    t_size = None                      # max length of the decoder input
    # Lower-triangular causal mask (Section 9.7, Figure 9.9c)
    causality_mask = None              # (t, t)
    # Combine causal and padding masks (Figure 9.11)
    look_ahead_mask = None             # (batch, 1, t, t)
    look_ahead_mask = None             # Typecast to float: 0. for masked positions, 1. for others

    ### END CODE HERE ###

    return look_ahead_mask

In [ ]:
x = tf.random.uniform((1, 3))
temp = create_look_ahead_mask(x)
print(temp)

**Expected Output**
```
tf.Tensor(
[[[[1. 0. 0.]
   [1. 1. 0.]
   [1. 1. 1.]]]], shape=(1, 1, 3, 3), dtype=float32)
```

In [ ]:
class Transformer(keras.Model):
    def __init__(self, *, num_layers, d_model, num_heads, dff, input_vocab_size,
                 target_vocab_size, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(num_layers=num_layers, d_model=d_model,
                               num_heads=num_heads, dff=dff,
                               input_vocab_size=input_vocab_size, dropout_rate=dropout_rate)

        self.decoder = Decoder(num_layers=num_layers, d_model=d_model,
                               num_heads=num_heads, dff=dff,
                               target_vocab_size=target_vocab_size, dropout_rate=dropout_rate)

        self.final_layer = keras.layers.Dense(target_vocab_size)

        self.last_attn_scores = None

    def call(self, inputs):
        # Keras models prefer if you pass all your inputs in the first argument

        inp, tar = inputs

        ### START CODE HERE ###

        # Create padding mask from encoder input (Section 9.7)
        enc_padding_mask = dec_padding_mask = None
        # Create causal + padding mask for decoder self-attention
        look_ahead_mask = None

        # Encoder forward pass (Section 9.3)
        enc_output = None

        # Decoder forward pass with cross-attention to encoder output
        dec_output = None

        # Final linear projection to vocabulary size
        final_output = None

        ### END CODE HERE ###

        self.last_attn_scores = self.decoder.last_attn_scores

        return final_output

In [ ]:
sample_transformer = Transformer(
    num_layers=2, d_model=512, num_heads=8, dff=2048,
    input_vocab_size=8500, target_vocab_size=8000)

temp_input = tf.random.uniform((64, 38), dtype=tf.int64, minval=0, maxval=200)
temp_target = tf.random.uniform((64, 36), dtype=tf.int64, minval=0, maxval=200)

fn_out = sample_transformer([temp_input, temp_target])

print(fn_out.shape)  # (batch_size, tar_seq_len, target_vocab_size)
print(fn_out[1,2,3:5])

**Expected Output**
```
(64, 36, 8000)
tf.Tensor([ 0.2744635  -0.17406853], shape=(2,), dtype=float32)
```

In [ ]:
attn_weights = sample_transformer.last_attn_scores
print(attn_weights.shape) # (batch, heads, target_seq, input_seq)


**Expected Output**
```
(64, 8, 36, 38)
```

## Set hyperparameters

To keep this example small and relatively fast, the values for `num_layers, d_model, dff` have been reduced.

The base model described in the [paper](https://arxiv.org/abs/1706.03762) used: `num_layers=6, d_model=512, dff=2048`.

In [ ]:
num_layers = 4
d_model = 128
dff = 512
num_heads = 8
dropout_rate = 0.1

## Optimizer

Use the Adam optimizer with a custom learning rate scheduler according to the formula in the [original paper](https://arxiv.org/abs/1706.03762). This warm-up schedule linearly increases the learning rate during the initial `warmup_steps`, then decreases it proportionally to the inverse square root of the step number:

$$\large{lr = d_{\text{model}}^{-0.5} \cdot \min(\text{step}^{-0.5},\; \text{step} \cdot \text{warmup\_steps}^{-1.5})}$$

Note: This learning rate schedule is defined in the original Transformer paper [Vaswani17] and is not separately discussed in the textbook.

In [ ]:
class CustomSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()

        self.d_model = d_model
        self.d_model = tf.cast(self.d_model, tf.float32)

        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, dtype=tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)

        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

In [ ]:
learning_rate = CustomSchedule(d_model)


if tf.__version__<'2.15':
    optimizer = keras.optimizers.legacy.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
else:
    optimizer = keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

In [ ]:
temp_learning_rate_schedule = CustomSchedule(d_model)

plt.figure(figsize=(6,3))
plt.plot(temp_learning_rate_schedule(tf.range(40000, dtype=tf.float32)))
plt.ylabel('Learning Rate')
plt.xlabel('Train Step')

## Loss and metrics

Since the target sequences are padded, it is important to apply a padding mask when calculating the loss so that padding tokens do not contribute to the training objective.

In [ ]:
def masked_loss(label, pred):
    mask = label != 0
    loss_object = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction='none')
    loss = loss_object(label, pred)

    mask = tf.cast(mask, dtype=loss.dtype)
    loss *= mask

    loss = tf.reduce_sum(loss)/tf.reduce_sum(mask)
    return loss


def masked_accuracy(label, pred):
    pred = tf.argmax(pred, axis=2)
    label = tf.cast(label, pred.dtype)
    match = label == pred

    mask = label != 0

    match = match & mask

    match = tf.cast(match, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(match)/tf.reduce_sum(mask)

## Training and checkpointing

In [ ]:
transformer = Transformer(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    input_vocab_size=tokenizers.pt.get_vocab_size().numpy(),
    target_vocab_size=tokenizers.en.get_vocab_size().numpy(),
    dropout_rate=dropout_rate)

In [ ]:
for (pt, en), en_labels in train_batches.take(1):
    break

print(pt.shape)
print(en.shape)
print(en_labels.shape)

The following test performs ```build``` the model.

In [ ]:
output = transformer((pt, en))

print(en.shape)
print(pt.shape)
print(output.shape)

In [ ]:
transformer.summary()

In [ ]:
transformer.compile(
    loss=masked_loss,
    optimizer=optimizer,
    metrics=[masked_accuracy],
    jit_compile=False,
)

In [ ]:
transformer.fit(train_batches,
                epochs=10,
                validation_data=val_batches)

The target is divided into `decoder_input` and `target_label`. `decoder_input` is passed as an input to the decoder. `target_label` is that same input shifted by 1: At each location in `decoder_input`, `target_label` contains the  next token that should be predicted.

For example, `sentence = 'SOS A lion in the jungle is sleeping EOS'` becomes:

* `decoder_input =  'SOS A lion in the jungle is sleeping'`
* `target_label = 'A lion in the jungle is sleeping EOS'`

A transformer is an auto-regressive model: it makes predictions one part at a time, and uses its output so far to decide what to do next.

During training this example uses teacher-forcing (like in the [text generation tutorial](https://www.tensorflow.org/text/tutorials/text_generation)). Teacher forcing is passing the true output to the next time step regardless of what the model predicts at the current time step.

As the model predicts each token, *self-attention* allows it to look at the previous tokens in the input sequence to better predict the next token.

To prevent the model from peeking at the expected output the model uses a look-ahead mask.

Portuguese is used as the input language and English is the target language.

### Run inference

The following steps are used for inference:

* Encode the input sentence using the Portuguese tokenizer (`tokenizers.pt`). This is the encoder input.
* The decoder input is initialized to the `[START]` token.
* Calculate the padding masks and the look ahead masks.
* The `decoder` then outputs the predictions by looking at the `encoder output` and its own output (self-attention).
* Concatenate the predicted token to the decoder input and pass it to the decoder.
* In this approach, the decoder predicts the next token based on the previous tokens it predicted.

Note: The model is optimized for _efficient training_ and makes a next-token prediction for each token in the output simultaneously. This is redundant during inference, and only the last prediction is used.  This model can be made more efficient for inference if you only calculate the last prediction when running in inference mode (`training=False`).

In [ ]:
class Translator(tf.Module):
    def __init__(self, tokenizers, transformer):
        self.tokenizers = tokenizers
        self.transformer = transformer

    def __call__(self, sentence, max_length=MAX_TOKENS):
        # input sentence is portuguese, hence adding the start and end token
        assert isinstance(sentence, tf.Tensor)
        if len(sentence.shape) == 0:
            sentence = sentence[tf.newaxis]

        sentence = self.tokenizers.pt.tokenize(sentence).to_tensor()

        encoder_input = sentence

        # As the output language is english, initialize the output with the
        # english start token.
        start_end = self.tokenizers.en.tokenize([''])[0]
        start = start_end[0][tf.newaxis]
        end = start_end[1][tf.newaxis]

        # `tf.TensorArray` is required here (instead of a python list) so that the
        # dynamic-loop can be traced by `tf.function`.
        output_array = tf.TensorArray(dtype=tf.int64, size=0, dynamic_size=True)
        output_array = output_array.write(0, start)

        for i in tf.range(max_length):
            output = tf.transpose(output_array.stack())
            predictions = self.transformer([encoder_input, output], training=False)

            # select the last token from the seq_len dimension
            predictions = predictions[:, -1:, :]  # (batch_size, 1, vocab_size)

            predicted_id = tf.argmax(predictions, axis=-1)

            # concatentate the predicted_id to the output which is given to the decoder
            # as its input.
            output_array = output_array.write(i+1, predicted_id[0])

            if predicted_id == end:
                break

        output = tf.transpose(output_array.stack())
        # output.shape (1, tokens)
        text = tokenizers.en.detokenize(output)[0]  # shape: ()

        tokens = tokenizers.en.lookup(output)[0]

        # `tf.function` prevents us from using the attention_weights that were
        # calculated on the last iteration of the loop. So recalculate them outside
        # the loop.
        self.transformer([encoder_input, output[:,:-1]], training=False)
        attention_weights = self.transformer.last_attn_scores

        return text, tokens, attention_weights

Note: This function uses an unrolled loop, not a dynamic loop. It generates `MAX_TOKENS` on every call. Refer to [NMT with attention](nmt_with_attention.ipynb) for an example implementation with a dynamic loop, which can be much more efficient.

Create an instance of this `Translator` class, and try it out a few times:

In [ ]:
translator = Translator(tokenizers, transformer)

In [ ]:
def print_translation(sentence, tokens, ground_truth):
    print(f'{"Input:":15s}: {sentence}')
    print(f'{"Prediction":15s}: {tokens.numpy().decode("utf-8")}')
    print(f'{"Ground truth":15s}: {ground_truth}')

In [ ]:
sentence = 'este é um problema que temos que resolver.'
ground_truth = 'this is a problem we have to solve .'

translated_text, translated_tokens, attention_weights = translator(
    tf.constant(sentence))
print_translation(sentence, translated_text, ground_truth)

In [ ]:
sentence = 'os meus vizinhos ouviram sobre esta ideia.'
ground_truth = 'and my neighboring homes heard about this idea .'

translated_text, translated_tokens, attention_weights = translator(
    tf.constant(sentence))
print_translation(sentence, translated_text, ground_truth)

In [ ]:
sentence = 'vou então muito rapidamente partilhar convosco algumas histórias de algumas coisas mágicas que aconteceram.'
ground_truth = "so i'll just share with you some stories very quickly of some magical things that have happened."

translated_text, translated_tokens, attention_weights = translator(
    tf.constant(sentence))
print_translation(sentence, translated_text, ground_truth)

## Attention plots

The `Translator` class returns a dictionary of attention maps you can use to visualize the internal working of the model:

In [ ]:
sentence = 'este é o primeiro livro que eu fiz.'
ground_truth = "this is the first book i've ever done."

translated_text, translated_tokens, attention_weights = translator(
    tf.constant(sentence))
print_translation(sentence, translated_text, ground_truth)

In [ ]:
def plot_attention_head(in_tokens, translated_tokens, attention):
    # The plot is of the attention when a token was generated.
    # The model didn't generate `<START>` in the output. Skip it.
    translated_tokens = translated_tokens[1:]

    ax = plt.gca()
    ax.matshow(attention)
    ax.set_xticks(range(len(in_tokens)))
    ax.set_yticks(range(len(translated_tokens)))

    labels = [label.decode('utf-8') for label in in_tokens.numpy()]
    ax.set_xticklabels(
        labels, rotation=90)

    labels = [label.decode('utf-8') for label in translated_tokens.numpy()]
    ax.set_yticklabels(labels)

In [ ]:
head = 0
# shape: (batch=1, num_heads, seq_len_q, seq_len_k)
attention_heads = tf.squeeze(attention_weights, 0)
attention = attention_heads[head]
attention.shape

In [ ]:
in_tokens = tf.convert_to_tensor([sentence])
in_tokens = tokenizers.pt.tokenize(in_tokens).to_tensor()
in_tokens = tokenizers.pt.lookup(in_tokens)[0]
in_tokens

In [ ]:
translated_tokens

In [ ]:
plt.figure(figsize=(4,4))

plot_attention_head(in_tokens, translated_tokens, attention)

In [ ]:
def plot_attention_weights(sentence, translated_tokens, attention_heads):
    in_tokens = tf.convert_to_tensor([sentence])
    in_tokens = tokenizers.pt.tokenize(in_tokens).to_tensor()
    in_tokens = tokenizers.pt.lookup(in_tokens)[0]
    in_tokens

    fig = plt.figure(figsize=(16, 8))

    for h, head in enumerate(attention_heads):
        ax = fig.add_subplot(2, 4, h+1)

        plot_attention_head(in_tokens, translated_tokens, head)

        ax.set_xlabel(f'Head {h+1}')

    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(10,10))

plot_attention_weights(sentence, translated_tokens,
                       attention_weights[0])

The model does okay on unfamiliar words. Neither "triceratops" or "encyclopedia" are in the input dataset and the model almost learns to transliterate them, even without a shared vocabulary:

In [ ]:
sentence = 'Eu li sobre triceratops na enciclopédia.'
ground_truth = 'I read about triceratops in the encyclopedia.'

translated_text, translated_tokens, attention_weights = translator(
    tf.constant(sentence))
print_translation(sentence, translated_text, ground_truth)

plt.figure(figsize=(10,10))

plot_attention_weights(sentence, translated_tokens, attention_weights[0])

## Export

That inference model is working, so next you'll export it as a `tf.saved_model`.

To do that, wrap it in yet another `tf.Module` sub-class, this time with a `tf.function` on the `__call__` method:

In [ ]:
class ExportTranslator(tf.Module):
    def __init__(self, translator):
        self.translator = translator

    @tf.function(input_signature=[tf.TensorSpec(shape=[], dtype=tf.string)])
    def __call__(self, sentence):
        (result,
         tokens,
         attention_weights) = self.translator(sentence, max_length=MAX_TOKENS)

        return result

In the above `tf.function` only the output sentence is returned. Thanks to the [non-strict execution](https://tensorflow.org/guide/intro_to_graphs) in `tf.function` any unnecessary values are never computed.

In [ ]:
translator = ExportTranslator(translator)

Since the model is decoding the predictions using `tf.argmax` the predictions are deterministic. The original model and one reloaded from its `SavedModel` should give identical predictions:

In [ ]:
translator('este é o primeiro livro que eu fiz.').numpy()

In [ ]:
tf.saved_model.save(translator, export_dir='translator')

In [ ]:
reloaded = tf.saved_model.load('translator')

In [ ]:
reloaded('este é o primeiro livro que eu fiz.').numpy()

## Summary

In this exercise you implemented and learned about:

* Positional encoding for injecting sequence order information (Section 9.4, Eq. 9.18)
* Scaled dot-product attention and multi-head attention (Sections 9.2, 9.6)
* Padding masks and causal masks for handling variable-length sequences and autoregressive decoding (Section 9.7)
* The complete Transformer encoder-decoder architecture for machine translation (Section 9.3)

This implementation follows the architecture of the [original paper](https://arxiv.org/abs/1706.03762). For further practice, consider:

* Using a different dataset to train the transformer.
* Creating the base Transformer configuration from the original paper (`num_layers=6, d_model=512, dff=2048`).
* Implementing beam search decoding (Section 9.1) for improved translation quality.

(c) 2026 S. W. Lee